# Notebook 154: 多様体学習と可視化 ― Manifold Learning & Visualization

## PCA・t-SNE・UMAPで高次元データの構造を見る

---

### このノートブックの位置づけ

**Embeddings シリーズ** の第5章として、高次元データを低次元空間に写像する「多様体学習」の手法群を学びます。

| 項目 | 内容 |
|------|------|
| 難易度 | ★★★☆☆ |
| 所要時間 | 120〜150分 |
| カテゴリ | Embeddings |

### 学習目標

- [ ] 多様体仮説を説明し、次元削減の必要性を理解できる
- [ ] PCA・t-SNE・UMAPの原理の違いを説明できる
- [ ] 各手法のパラメータが可視化結果に与える影響を体感できる
- [ ] 目的に応じた可視化手法の使い分けができる
- [ ] 高次元埋め込みの構造を2D/3Dで効果的に表現できる

### 前提知識

- ✅ Notebook 150（埋め込みの幾何学）
- ✅ 線形代数の基礎（固有値・固有ベクトル）
- ✅ 確率の基礎知識

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [多様体仮説とは？](#2-多様体仮説とは)
3. [PCA — 線形次元削減の基礎](#3-pca--線形次元削減の基礎)
4. [t-SNE — 局所構造の保存](#4-t-sne--局所構造の保存)
5. [UMAP — グローバル構造も保存](#5-umap--グローバル構造も保存)
6. [3手法の比較実験](#6-3手法の比較実験)
7. [埋め込みの可視化への応用](#7-埋め込みの可視化への応用)
8. [まとめ・チートシート・よくある間違い・自己評価クイズ](#8-まとめチートシートよくある間違い自己評価クイズ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# Section 1: 環境セットアップ
# ============================================================
# 必要なライブラリのインポート
# NumPy: 数値計算の基盤ライブラリ
import numpy as np
# matplotlib: グラフ描画ライブラリ
import matplotlib.pyplot as plt
# 3Dプロット用のツールキット
from mpl_toolkits.mplot3d import Axes3D
# seaborn: 統計的可視化ライブラリ
import seaborn as sns
# sklearn: 機械学習ライブラリ群
# PCA: 主成分分析（線形次元削減）
from sklearn.decomposition import PCA
# TSNE: t-分布型確率的近傍埋め込み（非線形次元削減）
# trustworthiness: 次元削減の品質評価指標
from sklearn.manifold import TSNE, trustworthiness
# make_swiss_roll: Swiss Roll データセット生成
# load_digits: 手書き数字データセット（8x8画像, 64次元）
from sklearn.datasets import make_swiss_roll, load_digits
# StandardScaler: 特徴量の標準化（平均0, 分散1に変換）
from sklearn.preprocessing import StandardScaler
# time: 実行時間の計測用
import time
# 警告メッセージを非表示にする
import warnings
warnings.filterwarnings('ignore')

# UMAP のインストール: pip install umap-learn
# UMAP: Uniform Manifold Approximation and Projection
# トポロジカルデータ解析に基づく非線形次元削減手法
try:
    import umap
    print("✅ UMAP ロード完了")
except ImportError:
    print("⚠️ umap-learn がインストールされていません")
    print("   pip install umap-learn を実行してください")

# ------------------------------------------------------------
# 日本語フォント設定（環境に応じて調整）
# ------------------------------------------------------------
def setup_japanese_font():
    """日本語フォントをmatplotlibに設定する関数"""
    # macOS: Hiragino Sans, Windows: MS Gothic, Linux: IPAGothic
    font_candidates = ['Hiragino Sans', 'Arial Unicode MS', 'IPAGothic', 'MS Gothic', 'sans-serif']
    plt.rcParams['font.family'] = font_candidates
    # マイナス記号の文字化け防止
    plt.rcParams['axes.unicode_minus'] = False
    # デフォルトの図サイズを設定
    plt.rcParams['figure.figsize'] = (10, 6)
    # デフォルトのフォントサイズを設定
    plt.rcParams['font.size'] = 11
    print("✅ 日本語フォント設定完了")

setup_japanese_font()

# ------------------------------------------------------------
# 再現性のためのシード
# ------------------------------------------------------------
np.random.seed(42)

# Seaborn のスタイル設定（白地にグリッド線）
sns.set_style('whitegrid')

print("✅ 環境セットアップ完了")
print(f"   NumPy version: {np.__version__}")

---

## 2. 多様体仮説とは？

### 2.1 高次元データは低次元多様体上にある

**多様体仮説 (Manifold Hypothesis)** とは、現実世界の高次元データは実際には低次元の **多様体 (manifold)** 上、またはその近傍に分布しているという仮説です。

直感的には：
- 画像は何百万次元のピクセル空間に存在するが、「自然な画像」はそのごく一部
- 自然言語の埋め込みは数百次元だが、意味的な構造はもっと少ない次元で説明できる

### 2.2 Swiss Roll の例

古典的な例として **Swiss Roll** データがあります。3次元空間に存在しますが、本質的には2次元の平面を「くるくる巻いた」構造です。

これを使って、なぜ線形手法（PCA）では多様体を正しく「ほどく」ことができないかを示します。

In [ ]:
# ============================================================
# Section 2: Swiss Roll データの生成と3D可視化
# ============================================================

# Swiss Roll データを生成
# make_swiss_roll: 3次元空間に渦巻き状のデータを生成する
# n_samples: データ点の数（多いほど滑らかな可視化）
# noise: ガウスノイズの標準偏差（多様体からのずれ）
# random_state: 再現性のためのシード値
# 戻り値: X (n_samples, 3) — 3D座標, color (n_samples,) — 多様体上の位置を示す色
n_samples = 2000
X_swiss, color_swiss = make_swiss_roll(n_samples=n_samples, noise=0.5, random_state=42)

# データの基本情報を表示
print(f"Swiss Roll データの形状: {X_swiss.shape}")
print(f"  3次元空間に {n_samples} 点が存在")
print(f"  しかし本質的には2次元の多様体（巻かれた平面）")
print(f"\n  X の範囲:")
for i, axis in enumerate(['x', 'y', 'z']):
    print(f"    {axis}: [{X_swiss[:, i].min():.1f}, {X_swiss[:, i].max():.1f}]")

# --- Plot 1: Swiss Roll の3D可視化 ---
# 2つの視点から同じデータを表示し、巻き構造を理解する
fig = plt.figure(figsize=(14, 6))

# 左パネル: 3D表示（アングル1 — 斜め上から見下ろす視点）
ax1 = fig.add_subplot(121, projection='3d')
# scatter: 散布図。c=color で多様体上の位置に応じて色付け
# cmap='Spectral': 赤→黄→緑→青の連続カラーマップ
ax1.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2],
            c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Swiss Roll（3D）\n色 = 多様体上の真の位置', fontsize=13)
# view_init: 視点の設定（elev=仰角, azim=方位角）
ax1.view_init(elev=10, azim=70)

# 右パネル: 3D表示（アングル2 — 正面から見た視点）
# この視点からだと巻きの断面構造がよく見える
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2],
            c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')
ax2.set_title('Swiss Roll（別視点）\n巻き構造が見える', fontsize=13)
ax2.view_init(elev=0, azim=0)

plt.tight_layout()
plt.show()

# 観察ポイントの解説
print("\n【観察ポイント】")
print("・色のグラデーションが多様体上の『本来の距離』を表している")
print("・3D空間で近い点でも、多様体上では遠い（巻きの内側と外側）")
print("・良い次元削減 = この色のグラデーションを2Dで再現できること")

---

## 3. PCA — 線形次元削減の基礎

### 3.1 PCA の数理

**主成分分析 (PCA: Principal Component Analysis)** は最も基本的な次元削減手法です。

**原理：分散最大化**
- データの分散が最も大きい方向（第1主成分）を見つける
- 第1主成分と直交する方向で、次に分散が大きい方向（第2主成分）を見つける
- これを繰り返す

**数学的には：共分散行列の固有値分解**

$$
\mathbf{C} = \frac{1}{n-1} \mathbf{X}^T \mathbf{X}
$$

$$
\mathbf{C} \mathbf{v}_i = \lambda_i \mathbf{v}_i
$$

- $\mathbf{v}_i$: 第 $i$ 固有ベクトル = 第 $i$ 主成分の方向
- $\lambda_i$: 第 $i$ 固有値 = その方向の分散

### 3.2 PCA の長所と限界

| 長所 | 限界 |
|------|------|
| 計算が高速 | 線形変換のみ |
| 解釈が容易 | 非線形構造を捉えられない |
| 寄与率で次元数を決定可能 | 多様体を「ほどく」ことができない |
| 再現性がある（決定的） | — |

In [ ]:
# ============================================================
# Section 3: PCA をスクラッチで実装
# ============================================================

class PCAFromScratch:
    """
    NumPy でゼロから実装する PCA。
    
    手順:
    1. データを中心化（平均を引く）
    2. 共分散行列を計算
    3. 固有値分解
    4. 上位 k 個の固有ベクトルを選択
    5. データを射影
    """
    
    def __init__(self, n_components=2):
        """PCA の初期化"""
        self.n_components = n_components
        self.components_ = None       # 主成分ベクトル
        self.explained_variance_ = None  # 各主成分の分散（固有値）
        self.explained_variance_ratio_ = None  # 寄与率
        self.mean_ = None             # データの平均
    
    def fit(self, X):
        """
        データから主成分を学習する。
        
        Parameters:
            X: numpy配列 (n_samples, n_features)
        """
        # ステップ1: データの中心化
        # 各特徴量の平均を0にする
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # ステップ2: 共分散行列の計算
        # C = (1/(n-1)) * X^T X
        n = X_centered.shape[0]
        cov_matrix = (X_centered.T @ X_centered) / (n - 1)
        
        # ステップ3: 固有値分解
        # 固有値（分散）と固有ベクトル（主成分方向）を取得
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # 固有値の大きい順にソート
        sorted_indices = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]
        
        # ステップ4: 上位 k 成分を選択
        self.components_ = eigenvectors[:, :self.n_components].T
        self.explained_variance_ = eigenvalues[:self.n_components]
        self.explained_variance_ratio_ = eigenvalues[:self.n_components] / np.sum(eigenvalues)
        
        return self
    
    def transform(self, X):
        """
        データを主成分空間に射影する。
        
        Parameters:
            X: numpy配列 (n_samples, n_features)
        Returns:
            numpy配列 (n_samples, n_components)
        """
        X_centered = X - self.mean_
        return X_centered @ self.components_.T
    
    def fit_transform(self, X):
        """fit と transform を一度に行う"""
        self.fit(X)
        return self.transform(X)


# --- スクラッチ実装と sklearn の結果を比較 ---
print("PCA スクラッチ実装 vs sklearn の比較")
print("=" * 60)

# スクラッチ実装
pca_scratch = PCAFromScratch(n_components=2)
X_pca_scratch = pca_scratch.fit_transform(X_swiss)

# sklearn 実装
pca_sklearn = PCA(n_components=2)
X_pca_sklearn = pca_sklearn.fit_transform(X_swiss)

# 結果の比較（符号の反転がありえるので絶対値で比較）
print(f"\n寄与率の比較:")
print(f"  スクラッチ: {pca_scratch.explained_variance_ratio_}")
print(f"  sklearn:    {pca_sklearn.explained_variance_ratio_}")

# 射影結果の差（符号を揃えて比較）
for i in range(2):
    # 符号の不定性を補正
    sign = np.sign(np.dot(X_pca_scratch[:, i], X_pca_sklearn[:, i]))
    diff = np.max(np.abs(X_pca_scratch[:, i] * sign - X_pca_sklearn[:, i]))
    print(f"  第{i+1}主成分の最大差: {diff:.2e}")

print("\n✅ スクラッチ実装と sklearn の結果がほぼ一致！")

In [ ]:
# ============================================================
# Plot 2: Swiss Roll の PCA 2D射影 — 構造が潰れることを示す
# ============================================================

# PCA で 3D → 2D に射影
# n_components=2: 2つの主成分に射影（2次元に削減）
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_swiss)

# PCA 全成分で寄与率を計算（3次元データなので3成分）
# 寄与率 = 各主成分が元データの分散の何%を説明するか
pca_full = PCA(n_components=3)
pca_full.fit(X_swiss)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左パネル: PCA の2D射影 ---
# 色が多様体上の本来の位置を表す → 混ざっていたら構造が壊れている
ax = axes[0]
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                     c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
ax.set_xlabel(f'第1主成分 (寄与率: {pca_2d.explained_variance_ratio_[0]:.1%})', fontsize=12)
ax.set_ylabel(f'第2主成分 (寄与率: {pca_2d.explained_variance_ratio_[1]:.1%})', fontsize=12)
ax.set_title('PCA による Swiss Roll の2D射影\n（線形射影では多様体がほどけない）', fontsize=13)
plt.colorbar(scatter, ax=ax, label='多様体上の位置')
ax.grid(True, alpha=0.3)

# --- 右パネル: 寄与率の棒グラフ ---
# 各主成分がどれだけの情報量（分散）を保持しているかを可視化
ax = axes[1]
components = ['PC1', 'PC2', 'PC3']
ratios = pca_full.explained_variance_ratio_
# 累積寄与率: 上位k成分で元データの何%を説明できるか
cumulative = np.cumsum(ratios)

# 棒グラフ（個別の寄与率）
bars = ax.bar(components, ratios, color=['#3498db', '#2ecc71', '#e74c3c'],
              edgecolor='black', linewidth=0.5, alpha=0.8, label='個別寄与率')
# 累積寄与率の折れ線グラフを重ねる
ax.plot(components, cumulative, 'ko-', linewidth=2, markersize=8, label='累積寄与率')

# 値をバーの上に表示して読みやすくする
for bar, val in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('寄与率', fontsize=12)
ax.set_title('PCA の寄与率（Explained Variance Ratio）', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# PCA の限界を解説
print("【観察ポイント】")
print("・PCA の2D射影では、色（多様体上の本来の位置）がぐちゃぐちゃに混ざっている")
print("・多様体上で遠い点（色が違う）が2D上で重なってしまう")
print("・PCA は分散最大の方向に射影するだけ → 巻き構造は無視される")
print("・これが線形次元削減の限界 → 非線形手法が必要")

---

## 4. t-SNE — 局所構造の保存

### 4.1 t-SNE の原理

**t-SNE (t-distributed Stochastic Neighbor Embedding)** は、高次元の局所的な構造を低次元に保存する非線形手法です。

**直感的な理解：**

1. **高次元空間**: 各点ペアの「近さ」を **ガウス分布** で確率化
   - 近い点ペアほど確率が高い
   - $p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}$

2. **低次元空間**: 各点ペアの「近さ」を **t分布** で確率化
   - $q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}}$

3. **最適化**: $p$ と $q$ の **KLダイバージェンス** を最小化
   - $\text{KL}(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$

### 4.2 なぜ t 分布を使うか？（Crowding Problem）

高次元空間では「中距離の点」が多数存在します。これを低次元に詰め込むと、点同士が **密集 (crowding)** してしまいます。

t分布はガウス分布より **裾が重い（heavy-tailed）** ため、低次元空間で中距離の点を適度に離すことができます。

In [ ]:
# ============================================================
# Section 4: t-SNE を Swiss Roll に適用
# ============================================================

# --- Plot 3: Swiss Roll の t-SNE 2D射影 ---
print("⏳ t-SNE を実行中（perplexity=30）...")
start_time = time.time()

# t-SNE の実行パラメータ:
# n_components=2: 2次元空間に射影
# perplexity=30: 「注目する近傍の数」の指標（通常 5-50）
#   - 条件付き確率のエントロピーに基づく有効近傍数
#   - 小さい → 局所的、大きい → 大域的
# n_iter=1000: 勾配降下法の最適化反復回数
# random_state=42: 再現性のためのシード値
# learning_rate='auto': 学習率の自動調整（sklearn 1.2+推奨）
# init='pca': PCA で初期化（ランダム初期化より安定）
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000,
            random_state=42, learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_swiss)

tsne_time = time.time() - start_time
print(f"✅ t-SNE 完了（{tsne_time:.2f}秒）")

# PCA と t-SNE を並べて比較（線形 vs 非線形の違いを明確にする）
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左パネル: PCA（再掲 — 比較のため）
ax = axes[0]
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                     c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
ax.set_title('PCA（線形）\n多様体がほどけていない', fontsize=13)
ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
plt.colorbar(scatter, ax=ax, label='多様体上の位置')
ax.grid(True, alpha=0.3)

# 右パネル: t-SNE（非線形 — 多様体をほどく）
ax = axes[1]
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
                     c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
ax.set_title('t-SNE（非線形）\n多様体がほどけている！', fontsize=13)
ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
plt.colorbar(scatter, ax=ax, label='多様体上の位置')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# t-SNE の成功を解説
print("\n【観察ポイント】")
print("・t-SNE は Swiss Roll の巻き構造をほどいて2Dに展開している")
print("・色のグラデーションが連続的に保存されている = 局所構造の保存")
print("・PCA ではぐちゃぐちゃだった構造が、t-SNE ではきれいに分離")

In [ ]:
# ============================================================
# Plot 4: perplexity の影響実験
# ============================================================

# perplexity = 「各点が注目する近傍の有効数」
# 小さい → 非常に局所的な構造のみ保存
# 大きい → より大域的な構造も考慮

perplexities = [5, 15, 30, 50, 100, 200]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

print("⏳ perplexity の影響を実験中...")

for idx, perp in enumerate(perplexities):
    print(f"   perplexity={perp}...", end=' ')
    start = time.time()
    
    # t-SNE を異なる perplexity で実行
    tsne_p = TSNE(n_components=2, perplexity=perp, n_iter=1000,
                  random_state=42, learning_rate='auto', init='pca')
    X_tsne_p = tsne_p.fit_transform(X_swiss)
    
    elapsed = time.time() - start
    print(f"{elapsed:.1f}秒")
    
    # プロット
    ax = axes[idx]
    ax.scatter(X_tsne_p[:, 0], X_tsne_p[:, 1],
              c=color_swiss, cmap='Spectral', s=8, alpha=0.7)
    ax.set_title(f'perplexity = {perp}', fontsize=13)
    ax.set_xlabel('t-SNE 1', fontsize=10)
    ax.set_ylabel('t-SNE 2', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('t-SNE: perplexity パラメータの影響\n（Swiss Roll, n=2000）',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("\n【perplexity の影響まとめ】")
print("・perplexity=5:   非常に局所的 → 細かいクラスタに分裂しやすい")
print("・perplexity=30:  標準的な値 → 多くの場合で良いバランス")
print("・perplexity=100: より大域的 → 構造が単純化される")
print("・推奨: データサイズの 5〜50 を目安に複数試す")

In [ ]:
# ============================================================
# t-SNE の注意点の実験: ランダム性の影響と実用上の制約
# ============================================================

# t-SNE を使う際に必ず知っておくべき重要な注意点をまとめる
# これらを知らないと、結果を誤解するリスクがある

# 注意点1: クラスタ間距離の非信頼性
# t-SNE はKLダイバージェンスを最小化する際、近い点の関係を優先する
# そのためクラスタ「内」の構造は信頼できるが、クラスタ「間」の距離は意味がない
print("【t-SNE の注意点】")
print("\n1. クラスタ間距離は意味をなさない")
print("   → t-SNE で離れたクラスタが『遠い』とは限らない")

# 注意点2: ランダム初期化による結果の変動
# 同じデータでもrandom_stateが違うと、全く異なる配置になることがある
# 特にクラスタの相対位置は実行ごとに変わる
print("\n2. ランダム性: 同じデータでも異なる結果")
print("   → 必ず random_state を設定すること")

# 注意点3: 計算量の制約
# 素朴な実装は O(N^2) — N=10000 でかなり遅くなる
# Barnes-Hut 近似で O(N log N) に改善可能（sklearn のデフォルト）
print("\n3. 計算コスト: O(N^2)")
print("   → 大規模データでは遅い（Barnes-Hut 近似で O(N log N) に改善可能）")

# 注意点4: ハイパーパラメータの重要性
# learning_rate が小さすぎると収束しない
# n_iter が少なすぎると途中で止まる
print("\n4. learning_rate と n_iter も重要")
print("   → 収束しない場合は n_iter を増やす")
print("   → learning_rate='auto' が sklearn 1.2+ で推奨")

---

## 5. UMAP — グローバル構造も保存

### 5.1 UMAP の原理

**UMAP (Uniform Manifold Approximation and Projection)** は、トポロジカルデータ解析に基づく手法です。

**直感的な理解：**

1. **局所的な距離構造の構築**: 各点について、k最近傍との距離を使って局所的な距離空間を定義
2. **ファジィ単体集合**: 各点の近傍グラフを「ファジィ」な辺の重みで表現。距離が近い点ほど辺の重みが大きい
3. **低次元への射影**: 低次元空間でも同様のファジィ集合構造を作り、高次元のものとのクロスエントロピーを最小化

### 5.2 t-SNE との比較

| 特性 | t-SNE | UMAP |
|------|-------|------|
| グローバル構造 | 保存しにくい | 比較的保存する |
| 計算速度 | 遅い O(N²) | 速い |
| 理論的基盤 | 確率分布の一致 | トポロジー |
| 新規データの変換 | 不可 | transform() が使える |
| 主なパラメータ | perplexity | n_neighbors, min_dist |

In [ ]:
# ============================================================
# Section 5: UMAP を Swiss Roll に適用
# ============================================================

# --- Plot 5: Swiss Roll の UMAP 2D射影 ---
print("⏳ UMAP を実行中...")
start_time = time.time()

# UMAP の実行
# n_neighbors: 局所構造を定義する近傍数（t-SNE の perplexity に類似）
# min_dist: 低次元空間での最小距離（小さい → 密集、大きい → 分散）
# metric: 距離関数（デフォルトは 'euclidean'）
umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.1,
                          n_components=2, random_state=42)
X_umap = umap_reducer.fit_transform(X_swiss)

umap_time = time.time() - start_time
print(f"✅ UMAP 完了（{umap_time:.2f}秒）")

# 3手法を並べて比較
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# PCA
ax = axes[0]
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                     c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
ax.set_title('PCA（線形）', fontsize=14)
ax.set_xlabel('PC1', fontsize=11)
ax.set_ylabel('PC2', fontsize=11)
ax.grid(True, alpha=0.3)

# t-SNE
ax = axes[1]
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
                     c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
ax.set_title('t-SNE（非線形）', fontsize=14)
ax.set_xlabel('t-SNE 1', fontsize=11)
ax.set_ylabel('t-SNE 2', fontsize=11)
ax.grid(True, alpha=0.3)

# UMAP
ax = axes[2]
scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1],
                     c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
ax.set_title('UMAP（非線形）', fontsize=14)
ax.set_xlabel('UMAP 1', fontsize=11)
ax.set_ylabel('UMAP 2', fontsize=11)
ax.grid(True, alpha=0.3)

plt.suptitle('Swiss Roll: 3手法の2D射影比較', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("\n【観察ポイント】")
print("・PCA:   巻き構造が潰れて色が混在")
print("・t-SNE: 多様体をほどけているが、グローバルな順序は崩れることがある")
print("・UMAP:  多様体をほどきつつ、色のグラデーション（グローバル構造）も保存")

In [ ]:
# ============================================================
# Plot 6: UMAP n_neighbors の影響
# ============================================================

# n_neighbors: 局所構造を定義する近傍数
# 小さい → 非常に局所的な構造を重視
# 大きい → より大域的な構造も考慮

n_neighbors_list = [5, 15, 50, 200]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

print("⏳ n_neighbors の影響を実験中...")

for idx, nn in enumerate(n_neighbors_list):
    print(f"   n_neighbors={nn}...", end=' ')
    start = time.time()
    
    # UMAP を異なる n_neighbors で実行
    reducer = umap.UMAP(n_neighbors=nn, min_dist=0.1,
                         n_components=2, random_state=42)
    X_umap_nn = reducer.fit_transform(X_swiss)
    
    elapsed = time.time() - start
    print(f"{elapsed:.1f}秒")
    
    ax = axes[idx]
    ax.scatter(X_umap_nn[:, 0], X_umap_nn[:, 1],
              c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
    ax.set_title(f'n_neighbors = {nn}', fontsize=13)
    ax.set_xlabel('UMAP 1', fontsize=10)
    ax.set_ylabel('UMAP 2', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('UMAP: n_neighbors パラメータの影響\n（Swiss Roll, min_dist=0.1）',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("\n【n_neighbors の影響まとめ】")
print("・n_neighbors=5:    非常に局所的 → 細かい断片に分裂しやすい")
print("・n_neighbors=15:   デフォルト値 → 多くのケースでバランスが良い")
print("・n_neighbors=50:   より大域的 → 滑らかな構造")
print("・n_neighbors=200:  大域構造重視 → 局所的な詳細が失われる")

In [ ]:
# ============================================================
# Plot 7: UMAP min_dist の影響
# ============================================================

# min_dist: 低次元空間での点間の最小距離
# 小さい → 密集した構造（クラスタが密）
# 大きい → 均一に分散した構造

min_dist_list = [0.0, 0.1, 0.5, 0.99]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

print("⏳ min_dist の影響を実験中...")

for idx, md in enumerate(min_dist_list):
    print(f"   min_dist={md}...", end=' ')
    start = time.time()
    
    # UMAP を異なる min_dist で実行
    reducer = umap.UMAP(n_neighbors=15, min_dist=md,
                         n_components=2, random_state=42)
    X_umap_md = reducer.fit_transform(X_swiss)
    
    elapsed = time.time() - start
    print(f"{elapsed:.1f}秒")
    
    ax = axes[idx]
    ax.scatter(X_umap_md[:, 0], X_umap_md[:, 1],
              c=color_swiss, cmap='Spectral', s=10, alpha=0.7)
    ax.set_title(f'min_dist = {md}', fontsize=13)
    ax.set_xlabel('UMAP 1', fontsize=10)
    ax.set_ylabel('UMAP 2', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('UMAP: min_dist パラメータの影響\n（Swiss Roll, n_neighbors=15）',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("\n【min_dist の影響まとめ】")
print("・min_dist=0.0:  点が密集 → クラスタの内部構造が見えやすい")
print("・min_dist=0.1:  デフォルト値 → バランスが良い")
print("・min_dist=0.5:  適度に分散 → 全体の配置が見やすい")
print("・min_dist=0.99: 均一に分散 → クラスタ構造が失われる")

---

## 6. 3手法の比較実験

### 6.1 MNIST Digits での比較

ここでは、より実用的なデータセットとして **sklearn の digits データセット** を使います。

- 8x8 ピクセルの手書き数字画像
- 64次元の特徴量ベクトル
- 10クラス（0-9）
- 1797サンプル

各手法が64次元データをどのように2Dに圧縮するかを比較し、定量的に評価します。

In [ ]:
# ============================================================
# Section 6: MNIST Digits データの準備
# ============================================================

# digits データセットのロード
digits = load_digits()
X_digits = digits.data      # (1797, 64) — 8x8画像を64次元に平坦化
y_digits = digits.target    # (1797,) — ラベル 0-9

print(f"Digits データセット:")
print(f"  サンプル数: {X_digits.shape[0]}")
print(f"  特徴量数:   {X_digits.shape[1]}（8x8ピクセル）")
print(f"  クラス数:   {len(np.unique(y_digits))}")
print(f"  各クラスのサンプル数: {np.bincount(y_digits)}")

# データのスケーリング
scaler = StandardScaler()
X_digits_scaled = scaler.fit_transform(X_digits)

# いくつかの画像を表示
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i in range(20):
    ax = axes[i // 10, i % 10]
    ax.imshow(digits.images[i], cmap='gray_r', interpolation='nearest')
    ax.set_title(f'{y_digits[i]}', fontsize=10)
    ax.axis('off')

plt.suptitle('Digits データセットのサンプル画像', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Plot 8: 3手法の比較（Digits データセット）
# ============================================================

# 色のマッピング: 数字 0-9 に対してカラーマップを使用
cmap_digits = plt.cm.tab10

# --- PCA ---
print("⏳ PCA を実行中...")
start = time.time()
pca_digits = PCA(n_components=2, random_state=42)
X_digits_pca = pca_digits.fit_transform(X_digits_scaled)
time_pca = time.time() - start
print(f"   PCA 完了: {time_pca:.3f}秒")

# --- t-SNE ---
print("⏳ t-SNE を実行中...")
start = time.time()
tsne_digits = TSNE(n_components=2, perplexity=30, n_iter=1000,
                   random_state=42, learning_rate='auto', init='pca')
X_digits_tsne = tsne_digits.fit_transform(X_digits_scaled)
time_tsne = time.time() - start
print(f"   t-SNE 完了: {time_tsne:.3f}秒")

# --- UMAP ---
print("⏳ UMAP を実行中...")
start = time.time()
umap_digits = umap.UMAP(n_neighbors=15, min_dist=0.1,
                         n_components=2, random_state=42)
X_digits_umap = umap_digits.fit_transform(X_digits_scaled)
time_umap = time.time() - start
print(f"   UMAP 完了: {time_umap:.3f}秒")

# --- 可視化 ---
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

methods = [
    ('PCA', X_digits_pca, time_pca),
    ('t-SNE', X_digits_tsne, time_tsne),
    ('UMAP', X_digits_umap, time_umap),
]

for idx, (name, X_2d, elapsed) in enumerate(methods):
    ax = axes[idx]
    
    # 各数字クラスを個別にプロット（凡例のため）
    for digit in range(10):
        mask = y_digits == digit
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                  c=[cmap_digits(digit)], s=15, alpha=0.7,
                  label=str(digit))
    
    ax.set_title(f'{name}（{elapsed:.3f}秒）', fontsize=14)
    ax.set_xlabel(f'{name} 1', fontsize=11)
    ax.set_ylabel(f'{name} 2', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(title='数字', fontsize=8, title_fontsize=9,
             loc='upper right', markerscale=1.5)

plt.suptitle('Digits データセット: PCA vs t-SNE vs UMAP\n（64次元 → 2次元）',
             fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\n【観察ポイント】")
print("・PCA:   一部のクラスタは分離するが、多くのクラスが重なっている")
print("・t-SNE: 各数字がきれいに分離されたクラスタを形成")
print("・UMAP:  t-SNE と同等以上の分離度で、より高速")

In [ ]:
# ============================================================
# 定量比較: Trustworthiness スコア
# ============================================================

# Trustworthiness: 低次元空間での近傍が、高次元空間でも近傍であるかを測る
# 値が 1.0 に近いほど、局所構造が保存されている

n_neighbors_eval = [5, 10, 15, 20, 30]

print("\n定量比較: Trustworthiness スコア")
print("=" * 60)
print("（1.0に近いほど局所構造が良く保存されている）")
print(f"\n{'k (近傍数)':<12} {'PCA':>10} {'t-SNE':>10} {'UMAP':>10}")
print("-" * 45)

results_trust = {'PCA': [], 't-SNE': [], 'UMAP': []}

for k in n_neighbors_eval:
    # 各手法の Trustworthiness を計算
    tw_pca = trustworthiness(X_digits_scaled, X_digits_pca, n_neighbors=k)
    tw_tsne = trustworthiness(X_digits_scaled, X_digits_tsne, n_neighbors=k)
    tw_umap = trustworthiness(X_digits_scaled, X_digits_umap, n_neighbors=k)
    
    results_trust['PCA'].append(tw_pca)
    results_trust['t-SNE'].append(tw_tsne)
    results_trust['UMAP'].append(tw_umap)
    
    # 最高スコアにマーカーをつける
    scores = {'PCA': tw_pca, 't-SNE': tw_tsne, 'UMAP': tw_umap}
    best = max(scores, key=scores.get)
    
    pca_marker = ' *' if best == 'PCA' else ''
    tsne_marker = ' *' if best == 't-SNE' else ''
    umap_marker = ' *' if best == 'UMAP' else ''
    
    print(f"k={k:<8} {tw_pca:>9.4f}{pca_marker} {tw_tsne:>9.4f}{tsne_marker} {tw_umap:>9.4f}{umap_marker}")

print("\n* = その k での最高スコア")

# Trustworthiness の可視化
fig, ax = plt.subplots(figsize=(10, 6))

colors_method = {'PCA': '#e74c3c', 't-SNE': '#3498db', 'UMAP': '#2ecc71'}
for method, scores in results_trust.items():
    ax.plot(n_neighbors_eval, scores, 'o-', linewidth=2, markersize=8,
            color=colors_method[method], label=method)

ax.set_xlabel('近傍数 k', fontsize=12)
ax.set_ylabel('Trustworthiness', fontsize=12)
ax.set_title('局所構造の保存度: Trustworthiness スコア\n（Digits データセット, 64D → 2D）', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.85, 1.01)

plt.tight_layout()
plt.show()

# 実行時間の比較
print("\n実行時間の比較:")
print(f"  PCA:   {time_pca:.3f}秒")
print(f"  t-SNE: {time_tsne:.3f}秒")
print(f"  UMAP:  {time_umap:.3f}秒")

In [ ]:
# ============================================================
# 比較まとめテーブル
# ============================================================

print("\n" + "=" * 75)
print("3手法の比較まとめ（Digits データセット）")
print("=" * 75)

# 各手法の Trustworthiness（k=10 での値）
k_ref = 10
idx_ref = n_neighbors_eval.index(k_ref)

print(f"\n{'手法':<8} {'実行時間':>10} {'Trust(k={0})'.format(k_ref):>15} {'局所保存':>10} {'大域保存':>10} {'再現性':>8}")
print("-" * 70)
print(f"{'PCA':<8} {time_pca:>9.3f}s {results_trust['PCA'][idx_ref]:>14.4f} {'△':>10} {'◎':>10} {'◎':>8}")
print(f"{'t-SNE':<8} {time_tsne:>9.3f}s {results_trust['t-SNE'][idx_ref]:>14.4f} {'◎':>10} {'△':>10} {'△':>8}")
print(f"{'UMAP':<8} {time_umap:>9.3f}s {results_trust['UMAP'][idx_ref]:>14.4f} {'◎':>10} {'○':>10} {'○':>8}")

print("\n◎=優秀  ○=良い  △=限定的")
print("\n【結論】")
print("・探索的分析: t-SNE または UMAP が適切")
print("・速度重視: PCA > UMAP > t-SNE")
print("・グローバル構造の保存: PCA > UMAP > t-SNE")
print("・新規データの変換: PCA, UMAP は可能。t-SNE は不可")

---

## 7. 埋め込みの可視化への応用

### 7.1 テキスト埋め込みの可視化

ここでは、`sentence-transformers` ライブラリを使って文の埋め込みを取得し、t-SNE / UMAP で可視化します。

ライブラリがインストールされていない場合は、TF-IDF ベクトルで代替します。

### 7.2 実践的な Tips

- **パイプライン**: 高次元 → PCA(50) → t-SNE/UMAP が推奨
  - PCA で事前にノイズを除去し、計算を高速化
- **再現性**: `random_state` を必ず設定
- **大規模データ**: サブサンプリング or UMAP（transform が使える）

In [ ]:
# ============================================================
# Section 7: 埋め込みの可視化（TF-IDF + t-SNE/UMAP）
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# サンプルテキスト: 4つのカテゴリに属する文
texts = [
    # スポーツ
    "The soccer team won the championship game last night",
    "Basketball players train hard every day for the season",
    "The tennis match lasted five sets in the tournament",
    "Swimming is an excellent exercise for the whole body",
    "The baseball pitcher threw a perfect game yesterday",
    "Football fans cheered loudly during the final quarter",
    "The marathon runner finished the race in record time",
    "Hockey teams compete fiercely in the winter league",
    # テクノロジー
    "Machine learning algorithms improve with more data",
    "The new smartphone features an advanced camera system",
    "Cloud computing has transformed how businesses operate",
    "Artificial intelligence is changing many industries",
    "The software update fixed several critical bugs",
    "Neural networks can recognize complex patterns in data",
    "Programming languages evolve to meet developer needs",
    "Cybersecurity threats continue to grow every year",
    # 料理
    "The chef prepared a delicious Italian pasta dish",
    "Baking bread requires patience and good ingredients",
    "Japanese sushi is known for its fresh fish and rice",
    "The recipe calls for olive oil and fresh herbs",
    "Chocolate cake is a popular dessert around the world",
    "Cooking with spices adds depth and flavor to meals",
    "The restaurant serves organic farm to table cuisine",
    "Grilling vegetables brings out their natural sweetness",
    # 科学
    "The experiment confirmed the theoretical prediction",
    "Climate change affects ecosystems around the globe",
    "DNA sequencing has revolutionized genetic research",
    "The telescope captured images of a distant galaxy",
    "Chemical reactions follow the laws of thermodynamics",
    "Quantum physics challenges our understanding of reality",
    "The vaccine was developed through years of research",
    "Marine biologists study coral reef ecosystems",
]

# カテゴリラベル
categories = (
    ['Sports'] * 8 +
    ['Technology'] * 8 +
    ['Cooking'] * 8 +
    ['Science'] * 8
)

# TF-IDF ベクトル化
vectorizer = TfidfVectorizer(max_features=200, stop_words='english')
X_tfidf = vectorizer.fit_transform(texts).toarray()

print(f"TF-IDF ベクトルの形状: {X_tfidf.shape}")
print(f"  {len(texts)} 文 x {X_tfidf.shape[1]} 特徴量")

# --- 高次元 → PCA(10) → t-SNE/UMAP のパイプライン ---
# ステップ1: PCA で次元削減（ノイズ除去 + 高速化）
pca_pre = PCA(n_components=min(10, X_tfidf.shape[1]), random_state=42)
X_pca_pre = pca_pre.fit_transform(X_tfidf)
print(f"PCA 前処理後: {X_pca_pre.shape}")
print(f"  累積寄与率: {sum(pca_pre.explained_variance_ratio_):.1%}")

# ステップ2a: t-SNE
tsne_text = TSNE(n_components=2, perplexity=8, n_iter=2000,
                 random_state=42, learning_rate='auto', init='pca')
X_text_tsne = tsne_text.fit_transform(X_pca_pre)

# ステップ2b: UMAP
umap_text = umap.UMAP(n_neighbors=8, min_dist=0.3,
                       n_components=2, random_state=42)
X_text_umap = umap_text.fit_transform(X_pca_pre)

print("\n✅ パイプライン完了")

In [ ]:
# ============================================================
# Plot 9: 埋め込み空間の2Dマップ（カテゴリ別色分け）
# ============================================================

category_colors_text = {
    'Sports': '#e74c3c',
    'Technology': '#3498db',
    'Cooking': '#2ecc71',
    'Science': '#9b59b6',
}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- 左: t-SNE ---
ax = axes[0]
for cat in category_colors_text:
    mask = [c == cat for c in categories]
    ax.scatter(X_text_tsne[mask, 0], X_text_tsne[mask, 1],
              c=category_colors_text[cat], s=100, alpha=0.8,
              label=cat, edgecolors='black', linewidth=0.5)

# テキストの一部をアノテーション
for i in range(0, len(texts), 4):  # 各カテゴリから2つだけ表示
    # テキストを短縮
    short_text = texts[i][:30] + '...'
    ax.annotate(short_text, (X_text_tsne[i, 0], X_text_tsne[i, 1]),
                textcoords='offset points', xytext=(5, 5),
                fontsize=7, alpha=0.8)

ax.set_title('t-SNE による文の埋め込み可視化\n（TF-IDF → PCA → t-SNE）', fontsize=13)
ax.set_xlabel('t-SNE 1', fontsize=11)
ax.set_ylabel('t-SNE 2', fontsize=11)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

# --- 右: UMAP ---
ax = axes[1]
for cat in category_colors_text:
    mask = [c == cat for c in categories]
    ax.scatter(X_text_umap[mask, 0], X_text_umap[mask, 1],
              c=category_colors_text[cat], s=100, alpha=0.8,
              label=cat, edgecolors='black', linewidth=0.5)

for i in range(0, len(texts), 4):
    short_text = texts[i][:30] + '...'
    ax.annotate(short_text, (X_text_umap[i, 0], X_text_umap[i, 1]),
                textcoords='offset points', xytext=(5, 5),
                fontsize=7, alpha=0.8)

ax.set_title('UMAP による文の埋め込み可視化\n（TF-IDF → PCA → UMAP）', fontsize=13)
ax.set_xlabel('UMAP 1', fontsize=11)
ax.set_ylabel('UMAP 2', fontsize=11)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n【観察ポイント】")
print("・同じカテゴリの文が近くにクラスタを形成している")
print("・TF-IDF は単語の頻度ベースなので、意味的類似度は完全ではない")
print("・sentence-transformers を使えばより意味的に正確な埋め込みが得られる")

In [ ]:
# ============================================================
# 実践的な Tips のまとめ
# ============================================================

print("=" * 65)
print("埋め込みの可視化: 実践的な Tips")
print("=" * 65)

print("""
【パイプラインの推奨】
  高次元データ（数百〜数千次元）
      ↓
  PCA で 50次元程度に削減（ノイズ除去 + 計算高速化）
      ↓
  t-SNE or UMAP で 2D/3D に可視化

【再現性の確保】
  ・random_state を必ず設定する
  ・t-SNE: random_state=42
  ・UMAP:  random_state=42
  ・PCA は決定的なので設定不要（ただしソルバーによっては必要）

【大規模データでの効率化】
  ・N > 10000 の場合:
    - t-SNE: Barnes-Hut 近似（sklearn のデフォルト）で O(N log N)
    - UMAP: そもそも高速。さらに transform() で新規データを変換可能
    - サブサンプリングして可視化し、全体の傾向を把握

【パラメータ選択のガイドライン】
  t-SNE:
    - perplexity: データサイズの 5〜50（デフォルト 30）
    - n_iter: 最低 250、推奨 1000+
    - learning_rate: 'auto'（sklearn 1.2+）
  
  UMAP:
    - n_neighbors: 5〜200（デフォルト 15）
    - min_dist: 0.0〜0.99（デフォルト 0.1）
    - クラスタ構造を見たい → min_dist 小さく
    - 全体の配置を見たい → min_dist 大きく
""")

---

## 8. まとめ・チートシート・よくある間違い・自己評価クイズ

### 8.1 まとめ

このノートブックでは、高次元データを低次元に可視化するための **多様体学習** を学びました。

1. **多様体仮説**: 高次元データは低次元多様体上に分布している
2. **PCA**: 線形射影。高速・解釈容易だが、非線形構造は捉えられない
3. **t-SNE**: 局所構造を保存。美しいクラスタ可視化だが、グローバル構造は失われやすい
4. **UMAP**: 局所+グローバル構造を保存。高速で transform() も使える
5. **パイプライン**: 高次元 → PCA → t-SNE/UMAP が実践的

### 8.2 チートシート

| 手法 | 原理 | 局所保存 | グローバル保存 | 速度 | 主なパラメータ |
|------|------|----------|---------------|------|---------------|
| PCA | 分散最大化（固有値分解） | △ | ◎ | ◎ | n_components |
| t-SNE | KLダイバージェンス最小化 | ◎ | △ | △ | perplexity, n_iter |
| UMAP | ファジィ単体集合の保存 | ◎ | ○ | ○ | n_neighbors, min_dist |

### 8.3 よくある間違い

#### 間違い 1: t-SNE のクラスタ間距離を意味あるものと解釈する

t-SNE は局所構造の保存に特化しているため、2D上でクラスタAとクラスタBが離れていても、高次元空間での距離を反映しているとは限りません。**クラスタの存在は信頼できますが、クラスタ間の距離は信頼できません**。

#### 間違い 2: perplexity / n_neighbors を吟味せずデフォルトのまま使う

パラメータはデータの性質（サンプル数、クラスタのサイズ、ノイズレベル）に大きく依存します。**複数の値で実験して結果を比較することが重要**です。Section 4, 5 で示したように、パラメータによって可視化結果は劇的に変わります。

#### 間違い 3: t-SNE に再現性がないことを忘れる（random_state 未設定）

t-SNE はランダム初期化に依存するため、`random_state` を設定しないと**毎回異なる結果**が得られます。論文やレポートでは必ず `random_state` を固定し、複数の random seed で結果の安定性を確認しましょう。

### 8.4 自己評価クイズ

---

**Q1.** 多様体仮説とは何ですか？Swiss Roll を例に説明してください。

<details>
<summary>回答を表示</summary>

多様体仮説とは、現実世界の高次元データは実際には **低次元の多様体（manifold）** 上、またはその近傍に分布しているという仮説です。Swiss Roll は3次元空間に存在しますが、本質的には **2次元の平面を巻いた構造** であり、データの「真の次元」は2です。良い次元削減手法は、この巻き構造を「ほどいて」元の2次元平面を復元できます。

</details>

---

**Q2.** PCA が Swiss Roll を正しく「ほどけない」理由を説明してください。

<details>
<summary>回答を表示</summary>

PCA は **線形射影** であり、分散が最大となる直交方向にデータを射影します。しかし Swiss Roll の構造は **非線形** であるため、どの直線方向に射影しても、巻きの内側と外側の点が重なってしまいます。PCA は多様体の「曲がり」を考慮できないため、非線形な多様体構造を保存できません。

</details>

---

**Q3.** t-SNE でなぜ t 分布（Student's t-distribution）を使うのですか？ガウス分布ではダメなのですか？

<details>
<summary>回答を表示</summary>

**Crowding Problem（混雑問題）** を解決するためです。高次元空間には「中距離の点」が大量に存在しますが、これらを低次元に詰め込むと点同士が密集してしまいます。t分布はガウス分布より **裾が重い（heavy-tailed）** ため、低次元空間で中距離の点に対してより大きな反発力を与え、適度に離すことができます。これにより、クラスタが明確に分離された可視化が得られます。

</details>

---

**Q4.** UMAP が t-SNE より優れている点を3つ挙げてください。

<details>
<summary>回答を表示</summary>

1. **計算速度**: UMAP は t-SNE より高速に動作します（特に大規模データで顕著）
2. **グローバル構造の保存**: t-SNE はクラスタ間距離を保存しませんが、UMAP はある程度グローバルな構造も保存します
3. **transform() が使える**: UMAP は学習済みモデルで新規データを変換できます。t-SNE は学習時のデータでしか使えません（全データで再計算が必要）

</details>

---

**Q5.** 10000次元の埋め込みベクトルを2Dで可視化したいとき、推奨されるパイプラインを述べてください。

<details>
<summary>回答を表示</summary>

推奨パイプライン:

1. **PCA で 50次元程度に削減**: ノイズの除去と計算の高速化が目的。累積寄与率 90% 以上を目安に次元数を決定
2. **t-SNE または UMAP で 2D に射影**: 局所構造を保存した可視化を得る
3. **random_state を設定**: 再現性を確保
4. **複数のパラメータで実験**: t-SNE なら perplexity、UMAP なら n_neighbors と min_dist を複数試す

直接 10000次元から t-SNE/UMAP を適用するのは避けましょう。計算コストが高く、ノイズの影響も受けやすくなります。

</details>

### 8.5 次のステップ

次の **Notebook 155: ベクトル検索とインデックス** では、高次元埋め込みベクトルを効率的に検索するための手法を学びます。

- 最近傍探索の基礎（brute-force vs 近似）
- FAISS / Annoy / ScaNN などのインデックスライブラリ
- プロダクション環境でのベクトル検索

---

## 参考文献

1. van der Maaten, L., & Hinton, G. (2008). *Visualizing Data using t-SNE*. JMLR, 9, 2579-2605.
2. McInnes, L., Healy, J., & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction*. arXiv:1802.03426.
3. Wattenberg, M., Viegas, F., & Johnson, I. (2016). *How to Use t-SNE Effectively*. Distill.
4. Jolliffe, I. T. (2002). *Principal Component Analysis*. Springer.